# CSE701 - 1η Εργασία | Επιλογή Α (Πρακτική)
## Starter Notebook

**Μην διαγράψετε τις επικεφαλίδες.** Συμπληρώστε κώδικα, σύντομες εξηγήσεις και validation checks (ελέγχους επικύρωσης).

### Στοιχεία
- Ονοματεπώνυμο/α: Πετρίδης Χρήστος
- ΑΜ: ics24199
- Ατομική / Ομαδική:  Ατομική

In [ ]:
import pandas as pd
from collections import defaultdict

DATA_PATH = '02_CSE701_Εργασία1_Clickstream.csv'
df = pd.read_csv(DATA_PATH)
df.head()

,event_id,timestamp,user_id,session_id,country,device,event_type,product_category,product_id,price_eur,quantity,response_ms,event_date,event_hour,is_purchase,revenue_eur,revenue_check
0,E0004910,2026-09-01 00:02:36,13347,S0003200,FR,desktop,view,Sports,P01623,13.11,0.0,255.0,2026-09-01,0.0,False,0.00,0.00
1,E0001353,2026-09-01 00:03:18,13219,S0002614,GR,mobile,view,Books,P01222,20.95,0.0,129.0,2026-09-01,0.0,False,0.00,0.00
2,E0004275,2026-09-01 00:04:28,13244,S0002837,FR,mobile,add_to_cart,Sports,P02449,60.75,0.0,358.0,2026-09-01,0.0,False,0.00,0.00
3,E0013513,2026-09-01 00:05:34,15421,S0006225,DE,desktop,view,Books,P01791,56.50,0.0,58.0,2026-09-01,0.0,False,0.00,0.00
4,E0005825,2026-09-01 00:07:45,15650,S0002791,BG,mobile,purchase,Home,P00208,35.81,2.0,180.0,2026-09-01,0.0,True,71.62,71.62


## A1. Data audit (Έλεγχος δεδομένων)
Καταγράψτε shape, dtypes, missing values, event counts και τους ελέγχους που θεωρείτε απαραίτητους. Μην κάνετε αλλαγές χωρίς να αιτιολογείτε γιατί.

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

DATA_PATH = '02_CSE701_Εργασία1_Clickstream.csv'
df = pd.read_csv(DATA_PATH)

print(f"Μέγεθος dataset: {df.shape}")
print("\nΤύποι μεταβλητών:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nΈλεγχος μοναδικότητας (Event IDs duplicates):", df['event_id'].duplicated().sum())

purchases = df[df['event_type'] == 'purchase']
print(f"\nΣυνολικά purchase events: {len(purchases)}")
missing_rev = purchases['revenue_eur'].isna().sum()
print(f"Purchases με κενό revenue_eur: {missing_rev}")

Μέγεθος dataset: (20000, 17)

Τύποι μεταβλητών:
event_id                str
timestamp               str
user_id               int64
session_id              str
country                 str
device                  str
event_type              str
product_category        str
product_id              str
price_eur           float64
quantity            float64
response_ms         float64
event_date              str
event_hour          float64
is_purchase            bool
revenue_eur         float64
revenue_check       float64
dtype: object

Missing values:
event_id              0
timestamp            55
user_id               0
session_id            0
country               0
device                0
event_type            0
product_category    179
product_id           80
price_eur           229
quantity             37
response_ms          18
event_date           55
event_hour           55
is_purchase           0
revenue_eur          16
revenue_check        16
dtype: int64

Έλεγχος μοναδικότητας (

Σύνοψη Data Audit & Αποφάσεις 

Προεπεξεργασίας Μέγεθος & Τύποι: 

Το dataset αποτελείται από $20.000$ γραμμές και $17$ στήλες. Τα event_id είναι 100% μοναδικά ($0$ duplicates).

Έλεγχοι Εγκυρότητας:

    Event types: 4 διακριτές τιμές (view: 11.580, search: 3.988, add_to_cart: 2.828, purchase: 1.604).
    
    Συσκευές & Χώρες: 3 συσκευές (mobile, desktop, tablet) και 8 ευρωπαϊκές χώρες (GR, DE, IT, FR, ES, CY, BG, RO).
    
    Consistency check: Στα purchases, επιβεβαιώθηκε ότι revenue_eur == price_eur * quantity.

Αποφάσεις Προεπεξεργασίας:

Χειρισμός Purchases χωρίς Revenue :

 Εντοπίστηκαν 16 αγορές με κενό revenue_eur (λόγω έλλειψης price ή quantity). Αποφασίστηκε η εξαίρεσή τους από τον υπολογισμό του top product ανά χώρα, καθώς η εκτίμηση πλασματικού τζίρου θα αλλοίωνε τα οικονομικά metrics.
 
 Χειρισμός Response Time :
 
  Υπάρχουν 18 κενές τιμές (NaN). Ο Mapper αγνοεί τις γραμμές αυτές ώστε ο κατανεμημένος υπολογισμός του μέσου όρου με ζεύγη (sum, count) να βασίζεται αποκλειστικά σε έγκυρες μετρήσεις. Τα υψηλά response_ms (outliers έως $24,8\text{ sec}$) διατηρούνται ως πραγματικά περιστατικά δικτύου/φόρτου (operational latency).

## Βοηθητική συνάρτηση Shuffle / Grouping
Μπορείτε να χρησιμοποιήσετε την παρακάτω συνάρτηση ή να γράψετε δική σας.

In [29]:
def shuffle_group(pairs):
    grouped = defaultdict(list)
    for key, value in pairs:
        grouped[key].append(value)
    return grouped


## A2. One-stage MapReduce (MapReduce ενός σταδίου)
**Ερώτημα:** αριθμός events ανά `(country, device)`.

1. Ορίστε το key/value. 2. Γράψτε mapper. 3. Κάντε Shuffle/Grouping. 4. Γράψτε reducer. 5. Επικυρώστε με pandas.

In [30]:
def mapper_event_count(row):
    key = (row['country'], row['device'])
    value = 1
    return (key, value)

map_output_a2 = [mapper_event_count(row) for _, row in df.iterrows()]

print("Συνολικά παραχθέντα ζεύγη:", len(map_output_a2))
print("Πρώτα 5 παραχθέντα ζεύγη (Key-Value pairs):", map_output_a2[:5])

Συνολικά παραχθέντα ζεύγη: 20000
Πρώτα 5 παραχθέντα ζεύγη (Key-Value pairs): [(('FR', 'desktop'), 1), (('GR', 'mobile'), 1), (('FR', 'mobile'), 1), (('DE', 'desktop'), 1), (('BG', 'mobile'), 1)]


In [31]:
grouped_a2 = shuffle_group(map_output_a2)


def reducer_event_count(grouped_data):
    results = []
    for (country, device), values in grouped_data.items():
        total_events = sum(values)
        results.append({
            'country': country,
            'device': device,
            'event_count': total_events
        })
    return pd.DataFrame(results)


event_counts_df = reducer_event_count(grouped_a2)
event_counts_df = event_counts_df.sort_values(by=['country', 'device']).reset_index(drop=True)
event_counts_df.head(10)

,country,device,event_count
0,BG,desktop,416
1,BG,mobile,901
2,BG,tablet,83
3,CY,desktop,508
4,CY,mobile,950
5,CY,tablet,103
6,DE,desktop,917
7,DE,mobile,1715
8,DE,tablet,153
9,ES,desktop,654


In [32]:
pandas_a2 = (
    df.groupby(['country', 'device'])
    .size()
    .reset_index(name='event_count')
    .sort_values(by=['country', 'device'])
    .reset_index(drop=True)
)

is_valid = event_counts_df.equals(pandas_a2)
print("Validation Check Α2:", "SUCCESS (100% ταύτιση)" if is_valid else "FAILED")

Validation Check Α2: SUCCESS (100% ταύτιση)


## A3. Two-stage pipeline (Αλυσίδα δύο σταδίων)
**Ερώτημα:** για κάθε χώρα, ποιο `product_id` έχει το μεγαλύτερο συνολικό `revenue_eur` από purchases;

**Job 1:** `(country, product_id) -> sum(revenue)`  
**Job 2:** `country -> max(product_revenue)`

In [33]:
valid_purchases = df[
    (df['event_type'] == 'purchase') & 
    (df['product_id'].notna()) & 
    (df['revenue_eur'].notna()) & 
    (df['revenue_eur'] > 0)
].copy()

print(f"Συνολικές έγκυρες αγορές για ανάλυση: {len(valid_purchases)}")
valid_purchases[['country', 'product_id', 'revenue_eur']].head()

Συνολικές έγκυρες αγορές για ανάλυση: 1581


,country,product_id,revenue_eur
4,BG,P00208,71.62
25,CY,P01117,17.69
38,DE,P02436,181.98
40,BG,P02489,6.00
59,FR,P01722,167.62


In [34]:
def mapper_j1(row):
    return ((row['country'], row['product_id']), row['revenue_eur'])

map_output_j1 = [mapper_j1(row) for _, row in valid_purchases.iterrows()]


grouped_j1 = shuffle_group(map_output_j1)


intermediate_job1 = {key: sum(values) for key, values in grouped_j1.items()}

print(f"Job 1 ολοκληρώθηκε: {len(intermediate_job1)} μοναδικά ζεύγη (country, product_id).")

list(intermediate_job1.items())[:5]

Job 1 ολοκληρώθηκε: 1506 μοναδικά ζεύγη (country, product_id).


[(('BG', 'P00208'), 71.62),
 (('CY', 'P01117'), 17.69),
 (('DE', 'P02436'), 181.98),
 (('BG', 'P02489'), 6.0),
 (('FR', 'P01722'), 167.62)]

In [35]:
def mapper_j2(item):
    (country, product_id), total_rev = item
    return (country, (product_id, total_rev))

map_output_j2 = [mapper_j2(item) for item in intermediate_job1.items()]


grouped_j2 = shuffle_group(map_output_j2)


top_products_list = []
for country, prod_tuples in grouped_j2.items():
    best_product, max_revenue = max(prod_tuples, key=lambda x: x[1])
    top_products_list.append({
        'country': country,
        'product_id': best_product,
        'total_revenue': round(max_revenue, 2)
    })

top_product_df = pd.DataFrame(top_products_list).sort_values(by='country').reset_index(drop=True)
top_product_df

,country,product_id,total_revenue
0,BG,P02460,1000.56
1,CY,P02196,650.24
2,DE,P00718,658.09
3,ES,P01819,376.02
4,FR,P00612,295.50
5,GR,P01043,1175.10
6,IT,P00027,387.06
7,RO,P02307,618.46


In [36]:
p_j1 = valid_purchases.groupby(['country', 'product_id'])['revenue_eur'].sum().reset_index()

p_j2 = p_j1.loc[p_j1.groupby('country')['revenue_eur'].idxmax()].reset_index(drop=True)
p_j2 = p_j2.rename(columns={'revenue_eur': 'total_revenue'})
p_j2['total_revenue'] = p_j2['total_revenue'].round(2)
p_j2 = p_j2.sort_values(by='country').reset_index(drop=True)

is_valid_a3 = top_product_df.equals(p_j2)
print("Validation Check Α3:", "SUCCESS (100% ταύτιση)" if is_valid_a3 else "FAILED")


Validation Check Α3: SUCCESS (100% ταύτιση)


## A4. Average response time ανά device
Η κατανεμημένη αναπαράσταση πρέπει να μεταφέρει **(sum, count)**. Εξηγήστε γιατί το average of averages μπορεί να είναι λάθος.

In [37]:
valid_response_df = df[df['response_ms'].notna()].copy()

def mapper_avg_response(row):
    return (row['device'], (row['response_ms'], 1))

map_output_a4 = [mapper_avg_response(row) for _, row in valid_response_df.iterrows()]
grouped_a4 = shuffle_group(map_output_a4)

print(f"Συνολικά ζεύγη που εκπέμφθηκαν: {len(map_output_a4)}")
print("Δείγμα shuffle για desktop (πρώτα 3 ζεύγη):", grouped_a4['desktop'][:3])

Συνολικά ζεύγη που εκπέμφθηκαν: 19982
Δείγμα shuffle για desktop (πρώτα 3 ζεύγη): [(255.0, 1), (58.0, 1), (151.0, 1)]


In [38]:
def reducer_avg_response(grouped_data):
    results = []
    for device, pairs in grouped_data.items():
        total_ms = sum(pair[0] for pair in pairs)
        total_count = sum(pair[1] for pair in pairs)
        avg_val = total_ms / total_count if total_count > 0 else 0.0
        results.append({
            'device': device,
            'avg_response_ms': round(avg_val, 2)
        })
    return pd.DataFrame(results)

avg_response_df = reducer_avg_response(grouped_a4)
avg_response_df = avg_response_df.sort_values(by='device').reset_index(drop=True)
avg_response_df


,device,avg_response_ms
0,desktop,292.22
1,mobile,278.74
2,tablet,269.67


In [39]:
pandas_a4 = (
    valid_response_df.groupby('device')['response_ms']
    .mean()
    .round(2)
    .reset_index(name='avg_response_ms')
    .sort_values(by='device')
    .reset_index(drop=True)
)

is_valid_a4 = avg_response_df.equals(pandas_a4)
print("Validation Check Α4:", "SUCCESS (100% ταύτιση)" if is_valid_a4 else "FAILED")

Validation Check Α4: SUCCESS (100% ταύτιση)


## A5. Scaling & architecture discussion
Απαντήστε συνοπτικά μέσα στο notebook ή παραπέμψτε σε συγκεκριμένη ενότητα της αναφοράς:
- Ποια δεδομένα περνούν από Shuffle;
    
    σε κάθε Job μεταφέρονται τα εξής:

    Job A2 (Event Counts): Μεταφέρονται όλα τα ζεύγη ((country, device), 1), ώστε να γίνει η καταμέτρηση των events ανά χώρα και συσκευή. Για παράδειγμα, αν υπάρχουν 20 δισεκατομμύρια events και δεν έχει προηγηθεί κάποια τοπική ομαδοποίηση, τότε θα χρειαστεί να μεταφερθούν αντίστοιχα 20 δισεκατομμύρια ζεύγη μέσω του δικτύου.

    Job A3 (Two-Stage Top Product):
        Job 1: Μεταφέρονται όλα τα ζεύγη ((country, product_id), revenue), προκειμένου να υπολογιστεί το συνολικό revenue κάθε προϊόντος ανά χώρα.
        Job 2: Εδώ μεταφέρεται πολύ μικρότερος όγκος δεδομένων, καθώς έχουν ήδη γίνει οι απαραίτητες συγκεντρώσεις στο προηγούμενο στάδιο. Συγκεκριμένα, μεταφέρονται ζεύγη της μορφής (country, (product_id, total_revenue)).

    Job A4 (Average Response): Μεταφέρονται όλα τα ζεύγη (device, (response_ms, 1)), ώστε να υπολογιστεί ο μέσος χρόνος απόκρισης για κάθε συσκευή.

- Πού θα βοηθούσε combiner (τοπικός συνδυαστής);

    Στο A2 (Event Counts): Μπορεί να μειώσει δραματικά το μέγεθος του Shuffle. Αντί ο Mapper να εκπέμπει εκατομμύρια φορές το ((country, device), 1), ο Combiner αθροίζει τοπικά και εκπέμπει μόνο ένα άθροισμα ανά κλειδί ανά κόμβο (το πολύ $8 \times 3 = 24$ ζεύγη ανά Worker).Στο A3 (Job 1 - Revenue sum): Επιτυγχάνει τεράστια μείωση αθροίζοντας τοπικά τα έσοδα ανά (country, product_id) πριν τη μετάδοση.Στο A4 (Average Response): Ο Combiner δεν μπορεί να υπολογίσει μέσο όρο (καθώς δεν είναι πράξη προσεταιριστική), αλλά μπορεί να αθροίσει τοπικά τα μερικά αθροίσματα και πλήθη: (device, (local_sum_ms, local_count)). Έτσι, αντί για δισεκατομμύρια events, στέλνονται μόνο 3 ζεύγη ανά worker κόμβο.

- Πού μπορεί να εμφανιστεί data skew (ανισοκατανομή);

    Χωρικό Skew (Country): Όπως φαίνεται και στο δείγμα μας, η Ελλάδα (GR) συγκεντρώνει το 32,14% όλων των events, ενώ χώρες όπως η Ρουμανία (RO) μόλις το 6,90%. Σε κατανεμημένο σύστημα, ο Reducer που αναλαμβάνει το partition της Ελλάδας θα επεξεργαστεί σχεδόν 5 φορές περισσότερα δεδομένα από τους άλλους.

    Skew Συσκευών (Device): Το mobile αντιστοιχεί σχεδόν στο 62% της κίνησης, ενώ το tablet μόλις στο 6%. Στο ερώτημα Α4, ο worker που διαχειρίζεται το κλειδί mobile θα δεχτεί τεράστιο φόρτο σε σχέση με τους υπόλοιπους.

    Skew Προϊόντων (Power-law distribution): Λίγα δημοφιλή/viral προϊόντα θα έχουν τεράστιο όγκο αγορών σε σχέση με τον «μακρύ ουρά» (long tail) του καταλόγου.

- Πώς θα κάνατε partitioning (διαμέριση);

    Σε επίπεδο αποθήκευσης (Storage Partitioning):
    Ιεραρχική διαμέριση στο Data Lake/Object Storage με βάση τον χρόνο και τη γεωγραφία
    π.χ.: /raw/clickstream/year=2026/month=09/day=01/country=GR/
    Αυτό επιτρέπει Partition Pruning: όταν εκτελείται ένα query για συγκεκριμένη ημέρα ή χώρα, διαβάζονται μόνο οι αντίστοιχοι φάκελοι και όχι ολόκληρο το dataset.

    Σε επίπεδο εκτέλεσης MapReduce (Shuffle Partitioner):
    Χρήση προσαρμοσμένου Partitioner με τεχνική Salting για την αντιμετώπιση του skew: προσθήκη ενός τυχαίου προθέματος/αριθμού στα «βαριά» κλειδιά (π.χ. GR_1, GR_2, GR_3) ώστε να μοιραστούν σε διαφορετικούς Reducers, και τελική συνένωση σε δεύτερο ελαφρύ πέρασμα.

- Τι αλλάζει αν έχουμε δισεκατομμύρια events;

    Οι βασικές αλλαγές που απαιτούνται είναι οι εξής:

    Αποθήκευση (Storage): Αντί να χρησιμοποιείται ένας απλός δίσκος ή ένα μεγάλο αρχείο CSV, χρειάζεται κατανεμημένη αποθήκευση, όπως το AWS S3, το Google Cloud Storage ή το HDFS. Επίσης, είναι προτιμότερο τα δεδομένα να αποθηκεύονται σε μορφή Apache Parquet, καθώς προσφέρει καλύτερη συμπίεση και πιο αποδοτική ανάγνωση σε σχέση με το CSV.

    Επεξεργασία (Compute): Η επεξεργασία δεν μπορεί πλέον να βασίζεται αποκλειστικά σε μία μηχανή με Python και Pandas. Χρειάζεται ένα κατανεμημένο εργαλείο, όπως το Apache Spark ή το MapReduce, το οποίο επιτρέπει την επεξεργασία των δεδομένων σε πολλούς υπολογιστές ταυτόχρονα. Με αυτόν τον τρόπο, η αποθήκευση και η επεξεργασία μπορούν να λειτουργούν ανεξάρτητα μεταξύ τους.

    Αξιοπιστία και Αντοχή σε Σφάλματα (Fault Tolerance & Reliability): Όταν ένα σύστημα αποτελείται από εκατοντάδες υπολογιστικούς κόμβους, είναι αναμενόμενο κάποιος δίσκος ή κάποιος κόμβος να παρουσιάσει βλάβη. Για να μη χάνονται τα δεδομένα και να μη διακόπτεται η επεξεργασία, χρησιμοποιούνται αντίγραφα των δεδομένων (replication), ενώ οι εργασίες που αποτυγχάνουν μπορούν να εκτελεστούν ξανά αυτόματα, χωρίς να χρειάζεται να επαναληφθεί ολόκληρη η διαδικασία.

    Εισαγωγή Δεδομένων (Ingestion): Για την ομαλή εισαγωγή μεγάλου όγκου δεδομένων, ειδικά όταν υπάρχουν ξαφνικές αυξήσεις στην κίνηση, είναι χρήσιμη η χρήση του Apache Kafka. Λειτουργεί ως ενδιάμεσο σύστημα που συγκεντρώνει και διαχειρίζεται τα εισερχόμενα δεδομένα, απορροφώντας τις αυξομειώσεις της κίνησης πριν αυτά αποθηκευτούν στη ζώνη Raw.

## A6. Export τελικών αποτελεσμάτων
Δημιουργήστε τα τρία απαιτούμενα CSV με ακριβώς αυτά τα ονόματα.

In [41]:
event_counts_df.to_csv('event_counts_country_device.csv', index=False)
top_product_df.to_csv('top_product_by_country.csv', index=False)
avg_response_df.to_csv('avg_response_by_device.csv', index=False)

print("Τα 3 CSV αρχεία δημιουργήθηκαν με επιτυχία στον τρέχοντα κατάλογο.")


Τα 3 CSV αρχεία δημιουργήθηκαν με επιτυχία στον τρέχοντα κατάλογο.


## Τελικό συμπέρασμα
Η πρακτική υλοποίηση απέδειξε ότι η λογική MapReduce (Mapper $\rightarrow$ Shuffle $\rightarrow$ Reducer) επιτρέπει την πλήρη αποσύνδεση και παράλληλη εκτέλεση βασικών αναλυτικών ερωτημάτων (καταμέτρηση, ομαδοποίηση, ιεραρχική μεγιστοποίηση και μέσος όρος μέσω ζευγών sum/count), παράγοντας αποτελέσματα που ταυτίζονται 100% με το pandas.Ωστόσο, σε περιβάλλον πραγματικής κλίμακας (δισεκατομμύρια events), το κύριο σημείο συμφόρησης (bottleneck) εντοπίζεται στη φάση Shuffle/Grouping και στις μεταφορές μέσω δικτύου (Network I/O), ιδίως λόγω του έντονου Data Skew (η Ελλάδα συγκεντρώνει το 32% της κίνησης και το mobile το 62%). Χωρίς τοπικούς Combiners και στρατηγική διαμέρισης (Partitioning/Salting), οι Reducers που αναλαμβάνουν τα «βαριά» κλειδιά θα μετατρέπονταν σε ακραίους stragglers, εξαντλώντας τη μνήμη RAM και καθυστερώντας ολόκληρο το κατανεμημένο cluster.  